In [57]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
from shapely import wkt as shapely_wkt
from google.colab import drive

In [59]:
import os

# token de acceso personal de GitHub (ver abajo cómo se guarda seguro)
from google.colab import userdata
TOKEN = userdata.get('GITHUB_TOKEN')

repo = f"https://{TOKEN}@github.com/Camilamop/RENABAP.git"
if not os.path.exists('/content/RENABAP'):
    os.system(f'git clone {repo} /content/RENABAP')

# 2. Ahora las rutas son LOCALES al entorno de Colab, limpias y rápidas
from pathlib import Path
RAIZ = Path("/content/RENABAP")
DIR_DATOS  = RAIZ / "data" / "raw"
DIR_SALIDA = RAIZ / "data" / "processed"

print("¿Existe raw?", DIR_DATOS.exists())

SecretNotFoundError: Secret GITHUB_TOKEN does not exist.

In [52]:
import platform, os, sys
print("Sistema del kernel:", platform.system())   # 'Windows' o 'Linux'
print("Directorio actual:", os.getcwd())
print("Python:", sys.executable)

Sistema del kernel: Linux
Directorio actual: /content
Python: /usr/bin/python3


In [35]:
#Diccionario de armonización: nombre_original -> nombre_núcleo

MAPEO = {
    2018: {
        "id_renabap": "id_renabap",
        "Nombre del barrio": "nombre_barrio",
        "Nombre de provincia": "provincia",
        "Nombre de departamentos/comuna": "departamento",
        "Localidad": "localidad",
        "Cantidad de familias": "cant_familias",
        "Tamaño (km2)": "superficie_km2",
        "Año de creación": "anio_creacion",
        "WKT": "wkt",
    },
    2022: {
        "renabap_id": "id_renabap",
        "nombre_barrio": "nombre_barrio",
        "provincia": "provincia",
       "departamento": "departamento",
        "localidad": "localidad",
        "cantidad_familias_aproximada": "cant_familias",
        "superficie_m2": "superficie_m2",
        "decada_de_creacion": "decada_creacion",
        "WKT": "wkt",
    },
    2023: {
        "id_renabap": "id_renabap",
        "nombre_barrio": "nombre_barrio",
        "provincia": "provincia",
        "departamento": "departamento",
        "localidad": "localidad",
        "cantidad_familias_aproximada": "cant_familias",
        "superficie_m2": "superficie_m2",
        "decada_de_creacion": "decada_creacion",
        "WKT": "wkt",
    },
}

# Columnas núcleo finales (orden estable para la tabla maestra)
NUCLEO = [
    "id_renabap", "registro", "nombre_barrio", "provincia", "departamento",
    "localidad", "cant_familias", "superficie_km2",
    "anio_creacion", "decada_creacion", "wkt",
]


In [36]:
def cargar_crudo(anio: int, ruta: Path) -> pd.DataFrame:
    df = pd.read_csv(ruta, dtype=str, keep_default_na=False, encoding="utf-8")
    df.columns = [c.strip() for c in df.columns]   # limpia espacios en headers
    return df


def normalizar_decada(serie_cruda: pd.Series) -> pd.Series: # convierte decadas a numero
    s = serie_cruda.astype(str).str.extract(r"(\d{4})")[0]
    return pd.to_numeric(s, errors="coerce").astype("Int64")


def armonizar(anio: int, df: pd.DataFrame) -> pd.DataFrame:
    mapa = MAPEO[anio]

    cols_presentes = {orig: nuevo for orig, nuevo in mapa.items() if orig in df.columns}
    out = df[list(cols_presentes.keys())].rename(columns=cols_presentes).copy()

    # registro (edición del registro)
    out["registro"] = anio

    # id como texto limpio
    out["id_renabap"] = out["id_renabap"].str.strip()

    # superficie a km2 (2023 viene en m2)
    if "superficie_m2" in out.columns:
        out["superficie_km2"] = pd.to_numeric(out["superficie_m2"], errors="coerce") / 1_000_000
        out = out.drop(columns=["superficie_m2"])
    elif "superficie_km2" in out.columns:
        out["superficie_km2"] = pd.to_numeric(out["superficie_km2"], errors="coerce")

    # cantidad de familias -> numérico
    if "cant_familias" in out.columns:
        out["cant_familias"] = pd.to_numeric(out["cant_familias"], errors="coerce").astype("Int64")

    # año de creación -> numérico (solo 2018 lo trae confiable)
    if "anio_creacion" in out.columns:
        anio_num = pd.to_numeric(out["anio_creacion"], errors="coerce")
        anio_num = anio_num.where((anio_num >= 1800) & (anio_num <= 2025))  # descarta 0 y basura
        out["anio_creacion"] = anio_num.astype("Int64")
    else:
        out["anio_creacion"] = pd.array([pd.NA] * len(out), dtype="Int64")

    # década de creación (armonizada a un entero tipo 1990)
    if "decada_cruda" in out.columns:
        out["decada_creacion"] = normalizar_decada(out["decada_cruda"])
        out = out.drop(columns=["decada_cruda"])
    else:
        # 2018 no trae década legible (viene codificada) -> se deriva del año
        out["decada_creacion"] = (out["anio_creacion"] // 10 * 10).astype("Int64")

    # localidad: garantizar la columna aunque la edición no la traiga (2022)
    if "localidad" not in out.columns:
        out["localidad"] = pd.NA

    # reordenar a esquema núcleo
    for c in NUCLEO:
        if c not in out.columns:
            out[c] = pd.NA
    return out[NUCLEO]


def a_geodataframe(df: pd.DataFrame) -> gpd.GeoDataFrame:
    """Parsea el WKT a geometría y arma un GeoDataFrame en EPSG:4326."""
    geom = df["wkt"].apply(lambda w: shapely_wkt.loads(w) if isinstance(w, str) and w.strip() else None)
    gdf = gpd.GeoDataFrame(df.drop(columns=["wkt"]), geometry=geom, crs=CRS_ORIGEN)
    return gdf

In [37]:
def main():
    piezas = []
    print("Cargando y armonizando los tres registros (2022 y 2023 oficiales)...\n")
    for anio, ruta in ARCHIVOS.items():
        crudo = cargar_crudo(anio, ruta)
        arm = armonizar(anio, crudo)
        piezas.append(arm)
        n_geom = arm["wkt"].apply(lambda w: isinstance(w, str) and w.strip() != "").sum()
        print(f"  {anio}: {len(arm):>5} barrios | geometrías no vacías: {n_geom}")

    maestra = pd.concat(piezas, ignore_index=True)
    gdf = a_geodataframe(maestra)
    invalidas = gdf.geometry.isna().sum()

    print(f"\nTabla maestra (largo): {len(maestra)} filas "
          f"{maestra['registro'].value_counts().sort_index().to_dict()}")
    print(f"Geometrías que no parsearon: {invalidas}")

    maestra.drop(columns=["wkt"]).to_csv(DIR_SALIDA / "renabap_maestra_largo.csv", index=False)
    for anio in ARCHIVOS:
        sub = gdf[gdf["registro"] == anio].dropna(subset=["geometry"])
        sub.to_file(DIR_SALIDA / "renabap_nacional.gpkg", layer=f"renabap_{anio}", driver="GPKG")

    print(f"\nExportado en: {DIR_SALIDA.resolve()}")
    print("  - renabap_maestra_largo.csv")
    print("  - renabap_nacional.gpkg (capas: RENABAP_2018 / 2022 / 2023)")

    print("\n--- Familias totales por registro (nacional) ---")
    print(maestra.groupby("registro")["cant_familias"].agg(["count", "sum"]))
    print("\n--- Cobertura de 'década de creación' por registro ---")
    print(maestra.groupby("registro")["decada_creacion"].apply(lambda s: round(s.notna().mean(), 3)))
    print("\n--- Cobertura de 'localidad' por registro ---")
    print(maestra.groupby("registro")["localidad"].apply(
        lambda s: round((s.notna() & (s.astype(str).str.strip() != "")).mean(), 3)))

    return maestra, gdf


if __name__ == "__main__":
    maestra, gdf = main()


Cargando y armonizando los tres registros (2022 y 2023 oficiales)...



FileNotFoundError: [Errno 2] No such file or directory: 'G:\\Mi unidad\\GITHUB\\RENABAP/data/raw/RENABAP_2018.csv'